# 3. TF-IDF transformacija i podela podataka

### 3.1 Učitavanje biblioteka i pretprocesiranih podataka

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import scipy.sparse

df = pd.read_csv('data/preprocessed_news.csv')
print(f'Ucitano {len(df)} clanaka')
print(f'Kolone: {list(df.columns)}')
df.head()

Ucitano 38483 clanaka
Kolone: ['title', 'text', 'subject', 'date', 'label', 'word_count', 'content', 'has_url', 'exclamation_count', 'caps_ratio', 'text_length', 'clean_content', 'tokens', 'final_text']


,title,text,subject,date,label,word_count,content,has_url,exclamation_count,caps_ratio,text_length,clean_content,tokens,final_text
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",News,"July 21, 2017",0,372,BREAKING: GOP Chairman Grassley Has Had Enoug...,False,0,0.047010,2191,BREAKING: GOP Chairman Grassley Has Had Enough...,"['BREAKING:', 'GOP', 'Chairman', 'Grassley', '...","BREAKING: GOP Chairman Grassley Enough, DEMAND..."
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,News,"May 7, 2016",0,504,Failed GOP Candidates Remembered In Hilarious...,True,1,0.040760,2895,Failed GOP Candidates Remembered In Hilarious ...,"['Failed', 'GOP', 'Candidates', 'Remembered', ...",Failed GOP Candidates Remembered Hilarious Moc...
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,News,"December 3, 2016",0,393,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,False,0,0.060217,2491,Mike Pences New DC Neighbors Are HILARIOUSLY T...,"['Mike', 'Pences', 'New', 'DC', 'Neighbors', '...",Mike Pences New DC Neighbors HILARIOUSLY Troll...
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,politicsNews,"October 6, 2017",1,97,California AG pledges to defend birth control ...,False,0,0.044669,694,California AG pledges to defend birth control ...,"['California', 'AG', 'pledge', 'defend', 'birt...",California AG pledge defend birth control insu...
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,politics,"Apr 25, 2017",0,157,AZ RANCHERS Living On US-Mexico Border Destroy...,False,0,0.067538,918,AZ RANCHERS Living On USMexico Border Destroy ...,"['AZ', 'RANCHERS', 'Living', 'USMexico', 'Bord...",AZ RANCHERS Living USMexico Border Destroy Nan...


### 3.2 Provera podataka pre transformacije

In [3]:
# Provera nedostajucih vrednosti u final_text
print(f'Nedostajuci final_text: {df["final_text"].isnull().sum()}')
print(f'Prazni final_text: {(df["final_text"].astype(str).str.strip() == "").sum()}')

# Uklanjanje redova bez teksta
df = df.dropna(subset=['final_text'])
df = df[df['final_text'].astype(str).str.strip() != '']
print(f'\nBroj clanaka nakon ciscenja: {len(df)}')

# Distribucija klasa
print(f'\nDistribucija klasa:')
print(df['label' if 'label' in df.columns else 'word_count'].value_counts())

Nedostajuci final_text: 0
Prazni final_text: 0

Broj clanaka nakon ciscenja: 38483

Distribucija klasa:
label
1    21191
0    17292
Name: count, dtype: int64


### 3.3 TF-IDF vektorizacija

In [4]:
# Kreiranje TF-IDF vektorizera
# max_features=10000 - ogranicava na 10000 najvaznijih reci
# ngram_range=(1,2) - koristi unigrame (jednu rec) i bigrame (par uzastopnih reci)
# max_df=0.95 - ignorise reci koje se pojavljuju u vise od 95% dokumenata
# min_df=2 - ignorise reci koje se pojavljuju u manje od 2 dokumenta

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    max_df=0.95,
    min_df=2
)

# Primena TF-IDF na tekst
X_tfidf = tfidf.fit_transform(df['final_text'].astype(str))

print(f'Dimenzije TF-IDF matrice: {X_tfidf.shape}')
print(f'Broj dokumenata: {X_tfidf.shape[0]}')
print(f'Broj feature-a (reci/bigrama): {X_tfidf.shape[1]}')
print(f'Tip matrice: {type(X_tfidf).__name__}')

Dimenzije TF-IDF matrice: (38483, 10000)
Broj dokumenata: 38483
Broj feature-a (reci/bigrama): 10000
Tip matrice: csr_matrix


### 3.4 Pregled najvažnijih TF-IDF feature-ova

In [5]:
# Top feature-i po prosecnom TF-IDF score-u
feature_names = tfidf.get_feature_names_out()
mean_tfidf = X_tfidf.mean(axis=0).A1

# Top 20 reci po TF-IDF
top_indices = mean_tfidf.argsort()[-20:][::-1]
print('Top 20 feature-a po prosecnom TF-IDF score-u:')
for i in top_indices:
    print(f'  {feature_names[i]:30s} {mean_tfidf[i]:.4f}')

Top 20 feature-a po prosecnom TF-IDF score-u:
  trump                          0.0544
  said                           0.0403
  president                      0.0215
  would                          0.0200
  state                          0.0176
  people                         0.0172
  house                          0.0165
  clinton                        0.0164
  obama                          0.0155
  reuters                        0.0147
  republican                     0.0144
  one                            0.0143
  new                            0.0142
  government                     0.0140
  donald                         0.0140
  white                          0.0139
  party                          0.0135
  election                       0.0129
  donald trump                   0.0128
  campaign                       0.0126


### 3.5 Podela na trening i test skup

In [6]:
# Labele
y = df['label' if 'label' in df.columns else 'word_count'].values

# Podela: 80% trening, 20% test
# stratify=y - obezbeduje istu distribuciju klasa u oba skupa
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Trening skup: {X_train.shape[0]} clanaka ({X_train.shape[0]/len(y)*100:.0f}%)')
print(f'Test skup:    {X_test.shape[0]} clanaka ({X_test.shape[0]/len(y)*100:.0f}%)')
print(f'\nDistribucija klasa u trening skupu:')
print(f'  Lazne:    {(y_train==0).sum()} ({(y_train==0).mean()*100:.1f}%)')
print(f'  Istinite: {(y_train==1).sum()} ({(y_train==1).mean()*100:.1f}%)')
print(f'\nDistribucija klasa u test skupu:')
print(f'  Lazne:    {(y_test==0).sum()} ({(y_test==0).mean()*100:.1f}%)')
print(f'  Istinite: {(y_test==1).sum()} ({(y_test==1).mean()*100:.1f}%)')

Trening skup: 30786 clanaka (80%)
Test skup:    7697 clanaka (20%)

Distribucija klasa u trening skupu:
  Lazne:    13833 (44.9%)
  Istinite: 16953 (55.1%)

Distribucija klasa u test skupu:
  Lazne:    3459 (44.9%)
  Istinite: 4238 (55.1%)


### 3.6 Čuvanje rezultata

In [7]:
import joblib

# Cuvanje TF-IDF matrica i labela
scipy.sparse.save_npz('data/X_train.npz', X_train)
scipy.sparse.save_npz('data/X_test.npz', X_test)
np.save('data/y_train.npy', y_train)
np.save('data/y_test.npy', y_test)

# Cuvanje TF-IDF vektorizera
joblib.dump(tfidf, 'data/tfidf_vectorizer.joblib')
print('Sacuvano:')
print(f'  data/X_train.npz  - trening matrica {X_train.shape}')
print(f'  data/X_test.npz   - test matrica {X_test.shape}')
print(f'  data/y_train.npy  - trening labele ({len(y_train)})')
print(f'  data/y_test.npy   - test labele ({len(y_test)})')
print(f'  data/tfidf_vectorizer.joblib - TF-IDF model')

Sacuvano:
  data/X_train.npz  - trening matrica (30786, 10000)
  data/X_test.npz   - test matrica (7697, 10000)
  data/y_train.npy  - trening labele (30786)
  data/y_test.npy   - test labele (7697)
  data/tfidf_vectorizer.joblib - TF-IDF model
